# OECD EPS

The OECD Environmental Policy Stringency Index (EPS) is a country-specific and internationally-comparable measure of the stringency of environmental policy.
Stringency is defined as the degree to which environmental policies put an explicit or implicit price on polluting or environmentally harmful behaviour.

The index is based on the degree of stringency of the following 13 environmental policy instruments, primarily related to climate and air pollution:

- 'CO2 Trading Scheme'
- 'Carbon dioxides (CO2) Tax'
- 'Diesel tax'
- 'Emission limit value NOx'
- 'Emission limit value PM'
- 'Emission limit value SOx'
- 'Emission limit value sulphur'
- 'Environmental Policy Stringency'
- 'Low-carbon R&D expenditures'
- 'Market based policies'
- 'Nitrogen Oxides (NOx) Tax'
- 'Non-market based policies'
- 'Renewable Energy Trading Scheme'
- 'Solar Energy support (Auctions & FITs)'
- 'Sulphur Oxides (SOx) Tax'
- 'Technology support policies'
- 'Wind Energy support (Auctions & FITs)'

The index ranges from 0 (not stringent) to 6 (highest degree of stringency) and covers 40 countries for the period 1990-2020.
[More information on OECD EPS dataset](https://doi.org/10.1787/90ab82e8-en)

> Last updated: 29 August 2022.

In [ ]:
import io
import os
import pandas as pd
import datetime as dt
import pandas_datareader.data as web

: 

DATA_SOURCE = 'oecd'
DATASET_NAME = 'EPS'

data_dir = "data"

data_raw_dir =  os.path.join(data_dir, DATA_SOURCE, "raw")
data_processed_dir = os.path.join(data_dir, DATA_SOURCE, "processed")

In [ ]:
# get data from OECD
start_time = dt.datetime(1990, 1, 1)
#end_time = dt.datetime(2022, 10, 31)
end_time = dt.date.today()
oecd_df = web.DataReader(DATASET_NAME, DATA_SOURCE, start_time, end_time)

In [ ]:
# unpivot dataframe
oecd_df = oecd_df.T.reset_index()
oecd_df = oecd_df.melt(id_vars=list(oecd_df.columns[:2]), var_name='year', value_name='value')

# convert np.datetime to year
oecd_df['year'] = [x.year for x in oecd_df['year']] 
# get list of poossible actions
#actions = list(oecd_df.Variable.unique())

# get list of OECD countries in this dataset
oecd_countries = list(oecd_df.Country.sort_values().unique())
oecd_countries

In [ ]:
# Map OECD countries to World Bank countries
# TODO: create data conversion table on /data/metadata
map_oecd_wb_country_codes = {}
wb_country_names = pd.read_csv('https://raw.githubusercontent.com/worldbank/SPI/master/01_raw_data/metadata/iso_codes.csv', usecols=['country','iso3c'])

direct_country_corresp = [i for i in wb_country_names['country'] if i in oecd_countries]
for country in direct_country_corresp:
  code = wb_country_names.loc[wb_country_names['country'] == country, 'iso3c'].values[0]
  map_oecd_wb_country_codes.update({country: code})


In [ ]:
# Make sure we have the mapped all countries in the dataset
assert len(map_oecd_wb_country_codes) == len(oecd_countries)

In [ ]:
missing_countries = [i for i in oecd_countries if i not in list(wb_country_names['country'])]
missing_countries

In [ ]:
country_manual_update = {"China (People's Republic of)": 'CHN', 
                         'Czech Republic': 'CZE',
                         'Korea':'KOR', 
                         'Russia':'RUS', 
                         'Slovak Republic':'SVK',
                         'Türkiye':'TUR',
                         'United Kingdom': 'GBR',
                         'United States': 'USA'
                         }

In [ ]:
# manual update for country names 
map_oecd_wb_country_codes.update(country_manual_update)

# Replace country name w/ country codes
oecd_df = oecd_df.replace(map_oecd_wb_country_codes)

# reshape oecd dataset to be like: year, country, intrument_1, intrument_2 ... intrument_n
## 1 row per country and year and 1 column per instrument 
oecd_df = oecd_df.pivot(index=['year', 'Country'], columns='Variable').reset_index()
idx_cols = ['year', 'country_code'] # repeated
oecd_df.columns =  [*idx_cols , *list(oecd_df.columns.droplevel(0)[2:])]
oecd_df

## Save
Save to .csv file

In [ ]:
output_filename = os.path.join(data_processed_dir, DATA_SOURCE + "_" + DATASET_NAME +".csv")
# Create the output directory if needed
if not os.path.exists(data_processed_dir):
    os.makedirs(data_processed_dir)
oecd_df.to_csv(output_filename, encoding='utf-8', index=False)